In [1]:
import numpy as np
import pandas as pd
import json

# Dataset

### Data Exploratory

In [2]:
df = pd.read_json("data/ner.json", lines=True)
df = df[['content', 'annotation']]
# df['content'] = df['content'].str.replace("\n", " ")
df

,content,annotation
0,Abhishek Jha\nApplication Development Associat...,"[{'label': ['Skills'], 'points': [{'start': 12..."
1,Afreen Jamadar\nActive member of IIIT Committe...,"[{'label': ['Email Address'], 'points': [{'sta..."
2,"Akhil Yadav Polemaina\nHyderabad, Telangana - ...","[{'label': ['Skills'], 'points': [{'start': 37..."
3,Alok Khandai\nOperational Analyst (SQL DBA) En...,"[{'label': ['Skills'], 'points': [{'start': 80..."
4,Ananya Chavan\nlecturer - oracle tutorials\n\n...,"[{'label': ['Degree'], 'points': [{'start': 20..."
...,...,...
215,"Mansi Thanki\nStudent\n\nJamnagar, Gujarat - E...","[{'label': ['College Name'], 'points': [{'star..."
216,Anil Kumar\nMicrosoft Azure (Basic Management)...,"[{'label': ['Location'], 'points': [{'start': ..."
217,Siddharth Choudhary\nMicrosoft Office Suite - ...,"[{'label': ['Skills'], 'points': [{'start': 78..."
218,Valarmathi Dhandapani\nInvestment Banking Oper...,"[{'label': ['Skills'], 'points': [{'start': 92..."


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220 entries, 0 to 219
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   content     220 non-null    object
 1   annotation  220 non-null    object
dtypes: object(2)
memory usage: 3.6+ KB


In [4]:
first_record = df.iloc[0]
first_record

,0
content,Abhishek Jha\nApplication Development Associat...
annotation,"[{'label': ['Skills'], 'points': [{'start': 12..."


In [5]:
print(first_record.content)

Abhishek Jha
Application Development Associate - Accenture

Bengaluru, Karnataka - Email me on Indeed: indeed.com/r/Abhishek-Jha/10e7a8cb732bc43a

• To work for an organization which provides me the opportunity to improve my skills
and knowledge for my individual and company's growth in best possible ways.

Willing to relocate to: Bangalore, Karnataka

WORK EXPERIENCE

Application Development Associate

Accenture -

November 2017 to Present

Role: Currently working on Chat-bot. Developing Backend Oracle PeopleSoft Queries
for the Bot which will be triggered based on given input. Also, Training the bot for different possible
utterances (Both positive and negative), which will be given as
input by the user.

EDUCATION

B.E in Information science and engineering

B.v.b college of engineering and technology -  Hubli, Karnataka

August 2013 to June 2017

12th in Mathematics

Woodbine modern school

April 2011 to March 2013

10th

Kendriya Vidyalaya

April 2001 to March 2011

SKILLS

C (Less

In [6]:
first_record.annotation

[{'label': ['Skills'],
  'points': [{'start': 1295,
    'end': 1621,
    'text': '\n• Programming language: C, C++, Java\n• Oracle PeopleSoft\n• Internet Of Things\n• Machine Learning\n• Database Management System\n• Computer Networks\n• Operating System worked on: Linux, Windows, Mac\n\nNon - Technical Skills\n\n• Honest and Hard-Working\n• Tolerant and Flexible to Different Situations\n• Polite and Calm\n• Team-Player'}]},
 {'label': ['Skills'],
  'points': [{'start': 993,
    'end': 1153,
    'text': 'C (Less than 1 year), Database (Less than 1 year), Database Management (Less than 1 year),\nDatabase Management System (Less than 1 year), Java (Less than 1 year)'}]},
 {'label': ['College Name'],
  'points': [{'start': 939, 'end': 956, 'text': 'Kendriya Vidyalaya'}]},
 {'label': ['College Name'],
  'points': [{'start': 883, 'end': 904, 'text': 'Woodbine modern school'}]},
 {'label': ['Graduation Year'],
  'points': [{'start': 856, 'end': 860, 'text': '2017\n'}]},
 {'label': ['College 

### Create Entities Col

In [7]:
def merge_overlap_entities(list_of_entities):
    """
    list_of_entities -- a list containing entity spans, each span is (start, end, label)

    Merges overlapping entities:
      - If same label: merge them into one.
      - If different labels: keep the longer one or prioritize the later.
    """

    entities = sorted(list_of_entities, key=lambda x: x[0])  # sort by start position
    merged_entities = []

    for ent in entities:
        if not merged_entities:
            merged_entities.append(ent)
        else:
            last_ent = merged_entities[-1]

            # If overlap
            if ent[0] <= last_ent[1]:

                # Same label → merge
                if last_ent[2] == ent[2]:
                    end = max(ent[1], last_ent[1])
                    merged_entities[-1] = (last_ent[0], end, last_ent[2])

                # Different labels → pick rule-based
                else:
                    if last_ent[1] > ent[1]:  # previous longer
                        merged_entities[-1] = last_ent
                    else:  # prioritize later one
                        merged_entities[-1] = (last_ent[0], ent[1], ent[2])

            # No overlap
            else:
                merged_entities.append(ent)

    return merged_entities

In [8]:
def get_entities(df):
    entities = []

    for i in range(len(df)):
        entities_each_record = []

        for annot in df['annotation'][i]:
            try:
                ent = annot['label'][0]
                start = annot['points'][0]['start']
                end = annot['points'][0]['end'] + 1  # plus 1 to get full text[start:end+1)
                entities_each_record.append((start, end, ent))
            except Exception as e:
                print(f"Error at record {i}: {e}")
                continue

        merged_entities_each_record = merge_overlap_entities(entities_each_record)
        entities.append(merged_entities_each_record)

    return entities

In [9]:
entities = get_entities(df)
len(entities), entities[0]

Error at record 81: list index out of range
Error at record 167: list index out of range


(220,
 [(0, 12, 'Name'),
  (13, 46, 'Designation'),
  (49, 58, 'Companies worked at'),
  (60, 69, 'Location'),
  (95, 146, 'Email Address'),
  (372, 405, 'Designation'),
  (407, 416, 'Companies worked at'),
  (727, 770, 'Designation'),
  (771, 814, 'College Name'),
  (856, 861, 'Graduation Year'),
  (883, 905, 'College Name'),
  (939, 957, 'College Name'),
  (993, 1154, 'Skills'),
  (1295, 1622, 'Skills')])

In [10]:
df['entities'] = entities
df

,content,annotation,entities
0,Abhishek Jha\nApplication Development Associat...,"[{'label': ['Skills'], 'points': [{'start': 12...","[(0, 12, Name), (13, 46, Designation), (49, 58..."
1,Afreen Jamadar\nActive member of IIIT Committe...,"[{'label': ['Email Address'], 'points': [{'sta...","[(0, 14, Name), (62, 68, Location), (104, 148,..."
2,"Akhil Yadav Polemaina\nHyderabad, Telangana - ...","[{'label': ['Skills'], 'points': [{'start': 37...","[(0, 21, Name), (22, 31, Location), (65, 117, ..."
3,Alok Khandai\nOperational Analyst (SQL DBA) En...,"[{'label': ['Skills'], 'points': [{'start': 80...","[(0, 12, Name), (13, 51, Designation), (54, 60..."
4,Ananya Chavan\nlecturer - oracle tutorials\n\n...,"[{'label': ['Degree'], 'points': [{'start': 20...","[(0, 13, Name), (14, 22, Designation), (24, 41..."
...,...,...,...
215,"Mansi Thanki\nStudent\n\nJamnagar, Gujarat - E...","[{'label': ['College Name'], 'points': [{'star...","[(0, 12, Name), (13, 20, Designation), (22, 30..."
216,Anil Kumar\nMicrosoft Azure (Basic Management)...,"[{'label': ['Location'], 'points': [{'start': ...","[(0, 10, Name), (11, 45, Designation), (47, 52..."
217,Siddharth Choudhary\nMicrosoft Office Suite - ...,"[{'label': ['Skills'], 'points': [{'start': 78...","[(0, 19, Name), (20, 51, Designation), (53, 62..."
218,Valarmathi Dhandapani\nInvestment Banking Oper...,"[{'label': ['Skills'], 'points': [{'start': 92...","[(0, 21, Name), (22, 56, Designation), (57, 66..."


# Preprocessing

### Cleaning + Preparing X, Y

In [11]:
def load_json_and_preprocess_based_on_annotation(json_file: str) -> list:
    data = []

    with open(json_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

        for line in lines:
            record = json.loads(line)  # convert JSON string -> dict
            content = record['content'].replace('\n', ' ')
            annot_list = record['annotation']

            entities_each_record = []

            for annot in annot_list:
                label_list = annot['label']
                point = annot['points'][0]

                # if label_list is not a list, then convert it to a list
                if not isinstance(label_list, list):
                    label_list = [label_list]

                for label in label_list:
                    start = point['start']
                    end = point['end']
                    text = point['text']

                    lstrip_diff = len(text) - len(text.lstrip())
                    rstrip_diff = len(text) - len(text.rstrip())
                    start += lstrip_diff
                    end -= rstrip_diff

                    entities_each_record.append((start, end+1, label))

            data.append((content, {'entities': entities_each_record}))

        return data

In [12]:
def preprocess_data_based_on_content(data: list) -> list:
    cleaned_data = []

    for content, entity_dict in data:
        entities = entity_dict['entities']

        cleaned_entities_each_record = []

        for start, end, label in entities:
            text_in_content = content[start:end]

            lstrip_diff = len(text_in_content) - len(text_in_content.lstrip())
            rstrip_diff = len(text_in_content) - len(text_in_content.rstrip())
            start += lstrip_diff
            end -= rstrip_diff

            cleaned_entities_each_record.append((start, end, label))

        cleaned_data.append((content, {'entities': cleaned_entities_each_record}))

    return cleaned_data

In [13]:
def map_token_to_entity(data: list):
    token_to_entity = []

    for content, entity_dict in data:
        words = content.split()
        labels = ['Empty'] * len(words)

        # Find word position in content
        current_pos = 0
        for i, word in enumerate(words):
            word_start = content.find(word, current_pos)
            word_end = word_start + len(word)
            current_pos = word_end

            # Map word to entity
            for (ent_start, ent_end, ent_label) in entity_dict.get("entities", []):
                if not (word_end <= ent_start or word_start >= ent_end):
                    labels[i] = ent_label
                    break

        token_to_entity.append(labels)

    return token_to_entity

In [14]:
data = preprocess_data_based_on_content(load_json_and_preprocess_based_on_annotation('data/ner.json'))
len(data)

220

In [15]:
entities = map_token_to_entity(data)
len(entities)

220

In [16]:
from collections import Counter

entities_flattened = [e for record in entities for e in record]
entity_counts = Counter(entities_flattened)

for entity, count in entity_counts.items():
    print(f"{entity}: {count}")

Name: 459
Designation: 1367
Empty: 99839
Companies worked at: 1145
Location: 456
Email Address: 327
College Name: 1213
Graduation Year: 253
Skills: 7676
Degree: 1095
Years of Experience: 89
UNKNOWN: 6


>**IMBALANCED CLASSES**

In [17]:
data[0]

("Abhishek Jha Application Development Associate - Accenture  Bengaluru, Karnataka - Email me on Indeed: indeed.com/r/Abhishek-Jha/10e7a8cb732bc43a  • To work for an organization which provides me the opportunity to improve my skills and knowledge for my individual and company's growth in best possible ways.  Willing to relocate to: Bangalore, Karnataka  WORK EXPERIENCE  Application Development Associate  Accenture -  November 2017 to Present  Role: Currently working on Chat-bot. Developing Backend Oracle PeopleSoft Queries for the Bot which will be triggered based on given input. Also, Training the bot for different possible utterances (Both positive and negative), which will be given as input by the user.  EDUCATION  B.E in Information science and engineering  B.v.b college of engineering and technology -  Hubli, Karnataka  August 2013 to June 2017  12th in Mathematics  Woodbine modern school  April 2011 to March 2013  10th  Kendriya Vidyalaya  April 2001 to March 2011  SKILLS  C (Le

In [18]:
data[0][0][95:145]

'Indeed: indeed.com/r/Abhishek-Jha/10e7a8cb732bc43a'

In [19]:
print(data[0][0].split())

['Abhishek', 'Jha', 'Application', 'Development', 'Associate', '-', 'Accenture', 'Bengaluru,', 'Karnataka', '-', 'Email', 'me', 'on', 'Indeed:', 'indeed.com/r/Abhishek-Jha/10e7a8cb732bc43a', '•', 'To', 'work', 'for', 'an', 'organization', 'which', 'provides', 'me', 'the', 'opportunity', 'to', 'improve', 'my', 'skills', 'and', 'knowledge', 'for', 'my', 'individual', 'and', "company's", 'growth', 'in', 'best', 'possible', 'ways.', 'Willing', 'to', 'relocate', 'to:', 'Bangalore,', 'Karnataka', 'WORK', 'EXPERIENCE', 'Application', 'Development', 'Associate', 'Accenture', '-', 'November', '2017', 'to', 'Present', 'Role:', 'Currently', 'working', 'on', 'Chat-bot.', 'Developing', 'Backend', 'Oracle', 'PeopleSoft', 'Queries', 'for', 'the', 'Bot', 'which', 'will', 'be', 'triggered', 'based', 'on', 'given', 'input.', 'Also,', 'Training', 'the', 'bot', 'for', 'different', 'possible', 'utterances', '(Both', 'positive', 'and', 'negative),', 'which', 'will', 'be', 'given', 'as', 'input', 'by', 'the'

In [20]:
print(entities[0])

['Name', 'Name', 'Designation', 'Designation', 'Designation', 'Empty', 'Companies worked at', 'Location', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Email Address', 'Email Address', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Designation', 'Designation', 'Designation', 'Companies worked at', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 

### Convert Y to Indices

In [21]:
unique_entities = set(entity for record in entities for entity in record)
entity_to_idx = {ent:idx for idx, ent in enumerate(unique_entities)}
idx_to_entity = {idx:ent for idx, ent in enumerate(unique_entities)}

entity_to_idx

{'Companies worked at': 0,
 'College Name': 1,
 'Designation': 2,
 'Location': 3,
 'UNKNOWN': 4,
 'Graduation Year': 5,
 'Degree': 6,
 'Skills': 7,
 'Email Address': 8,
 'Name': 9,
 'Years of Experience': 10,
 'Empty': 11}

In [22]:
def map_entity_to_idx(entities_all_records: list, entity_to_idx: dict) -> list:
    entity_indices_all_records = []

    for entities_each_record in entities_all_records:
        entity_indices_each_record = []

        for entity in entities_each_record:
            entity_indices_each_record.append(entity_to_idx[entity])

        entity_indices_all_records.append(entity_indices_each_record)

    return entity_indices_all_records

In [23]:
entity_indices = map_entity_to_idx(
    entities_all_records=entities,
    entity_to_idx=entity_to_idx
)

len(entity_indices)

220

In [24]:
print(entities[0])
print()
print(entity_indices[0])

['Name', 'Name', 'Designation', 'Designation', 'Designation', 'Empty', 'Companies worked at', 'Location', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Email Address', 'Email Address', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Designation', 'Designation', 'Designation', 'Companies worked at', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 'Empty', 

### Padding Y

In [25]:
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [26]:
# Pick the seq_length the same as seq_length for BERT model below
MAX_SEQ_LENGTH = 512

entity_indices_padded = pad_sequences(
    entity_indices,
    maxlen=MAX_SEQ_LENGTH,
    value=entity_to_idx["Empty"],
    padding='post',
    truncating='post',
    dtype='int32'
)

# padding -- 'pre'(pad at the first of sequence) and 'post'(pad at the end of sequence)
# value -- if seq_length < max_seq_length, then pad it with value ...
# truncating -- if seq_length > max_seq_length, then truncate at the first or at the end of sequence

In [27]:
entity_indices_padded.shape

(220, 512)

# Pretrained Tokenizer + Model

* `Fast` -- tokenizer written in Rust, a more optimal and faster way (default is written in Python)
* `TF` -- model written in TensorFlow (default is PyTorch model)

### 1. Variants of BERT model
- `BertModel`
    - original
    - 12 - 24 layers
    - 110M - 340M parameters
---
- `DistilBertModel`
    - a smaller version of BERT using knowledge distillation to keep the probability output the same as a large model to maintain 97% efficiency
    - 6 layers
    - 66M parameters
---
- `RobertaModel`
    - a more robust BERT with a better training approach: no "Next Sentence Prediction", dynamic masking, trained on a larger corpus
    - 12 - 24 layers
    - 125M - 355M parameters
---    
- `AlbertModel`
    - a lite BERT using parameter sharing + factorized embedding to reduce parameters
    - 12 - 24 layers
    - 12M - 18M parameters
---    
- `TinyBertModel`
    - extremely lightweight version of BERT, optimized for mobile/edge devices, also uses knowledge distillation
    - 4 - 6 layers
    - 14M - 66M parameters
---
- `MobileBertModel`
    - optimized for mobile inference, uses bottleneck structure to reduce parameters, but more layers to maintain efficiency
    - 24 layers
    - 25M parameters

### 2. Specific-Tasks for BERT model

- `BertModel`
    - returns contextualized embedding vector of each token, downstream to other tasks
    - tasks: feature extraction, ...
---
- `BertForTokenClassification`:
    - returns classification of each token
    - task: NER, POS, ...
---
- `BertForSequenceClassification`:
    - returns classification of whole sequence
    - tasks: sentiment analysis, ...
---
- `BertForQuestionAnswering`

- `BertForMaskedLM`

- `BertForNextSentencePrediction`

In [28]:
from transformers import DistilBertTokenizerFast, TFDistilBertForTokenClassification

**Pick model here:** https://huggingface.co/models?pipeline_tag=token-classification&library=tf&sort=downloads&search=distil

In [29]:
MODEL_NAME_FROM_HUGGINGFACE= "distilbert-base-uncased"
N_UNIQUE_ENTITIES = len(unique_entities)

model = TFDistilBertForTokenClassification.from_pretrained(MODEL_NAME_FROM_HUGGINGFACE, num_labels=N_UNIQUE_ENTITIES, from_pt=True)
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME_FROM_HUGGINGFACE)

# Freeze base model
model.distilbert.trainable = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForTokenClassification: ['vocab_layer_norm.weight', 'vocab_projector.bias', 'vocab_layer_norm.bias', 'vocab_transform.weight', 'vocab_projector.weight', 'vocab_transform.bias']
- This IS expected if you are initializing TF

### Model Exploratory

In [30]:
model.summary()

Model: "tf_distil_bert_for_token_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 distilbert (TFDistilBertMa  multiple                  66362880  
 inLayer)                                                        
                                                                 
 dropout_19 (Dropout)        multiple                  0         
                                                                 
 classifier (Dense)          multiple                  9228      
                                                                 
Total params: 66372108 (253.19 MB)
Trainable params: 9228 (36.05 KB)
Non-trainable params: 66362880 (253.15 MB)
_________________________________________________________________


In [31]:
for i in model.layers:
    print(f"{i.name}: {i.trainable}")

distilbert: False
dropout_19: True
classifier: True


In [32]:
classifier_layer = model.get_layer('classifier')
classifier_layer.weights

[<tf.Variable 'tf_distil_bert_for_token_classification/classifier/kernel:0' shape=(768, 12) dtype=float32, numpy=
 array([[-0.02286054, -0.01909198,  0.03298096, ...,  0.003943  ,
         -0.03092858,  0.02505971],
        [-0.03099955,  0.01869007, -0.00388006, ...,  0.02577671,
          0.01120206,  0.02923526],
        [ 0.02278301,  0.002283  , -0.00768174, ...,  0.02594946,
         -0.00053585,  0.00490467],
        ...,
        [-0.02288121, -0.00119092,  0.01011784, ...,  0.00261992,
          0.00288809, -0.01133754],
        [ 0.01089716, -0.0086719 ,  0.0156045 , ...,  0.01621933,
         -0.01459888, -0.00191578],
        [-0.00600209,  0.019675  , -0.01424886, ..., -0.0039912 ,
         -0.01150142, -0.01046909]], dtype=float32)>,
 <tf.Variable 'tf_distil_bert_for_token_classification/classifier/bias:0' shape=(12,) dtype=float32, numpy=array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)>]

In [33]:
dropout_layer = model.get_layer('dropout_19')
dropout_layer.rate = 0.0
dropout_layer.rate

0.0

In [34]:
distilbert_model = model.distilbert
embedding_layer = distilbert_model.embeddings

for w in embedding_layer.weights:
    print(f"{w.name:<100}: {w.shape}")

tf_distil_bert_for_token_classification/distilbert/embeddings/word_embeddings/weight:0              : (30522, 768)
tf_distil_bert_for_token_classification/distilbert/embeddings/position_embeddings/embeddings:0      : (512, 768)
tf_distil_bert_for_token_classification/distilbert/embeddings/LayerNorm/gamma:0                     : (768,)
tf_distil_bert_for_token_classification/distilbert/embeddings/LayerNorm/beta:0                      : (768,)


In [35]:
layer_weights = {}

for w in distilbert_model.weights:
    name = w.name.split('distilbert/')[1]

    if name.startswith("embeddings/"):
        # Create a list at first to store weights of that layer
        layer_name = "embedding_layer"
        if layer_name not in layer_weights:
            layer_weights[layer_name] = []

        # Append weights to that list
        clean_name = name.replace("embeddings/", "")
        layer_weights[layer_name].append((clean_name, w.shape))

    elif name.startswith('transformer/'):
        layer_idx = name.split('/')[1].split('_._')[1]
        layer_name = f"transformer_layer_{layer_idx}"
        if layer_name not in layer_weights:
            layer_weights[layer_name] = []

        clean_name = "/".join(name.split('/')[2:])
        layer_weights[layer_name].append((clean_name, w.shape))

In [36]:
for layer_name, weights in layer_weights.items():
    print(f"LAYER: {layer_name.upper()}")
    for name, shape in weights:
        print(f"\t{name:<70} {shape}")
    print('~'*100)

LAYER: EMBEDDING_LAYER
	word_weight:0                                                          (30522, 768)
	position_embeddings:0                                                  (512, 768)
	LayerNorm/gamma:0                                                      (768,)
	LayerNorm/beta:0                                                       (768,)
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
LAYER: TRANSFORMER_LAYER_0
	attention/q_lin/kernel:0                                               (768, 768)
	attention/q_lin/bias:0                                                 (768,)
	attention/k_lin/kernel:0                                               (768, 768)
	attention/k_lin/bias:0                                                 (768,)
	attention/v_lin/kernel:0                                               (768, 768)
	attention/v_lin/bias:0                                                 (768,)
	attention/out_lin/kernel:0          

### Tokenizer Exploratory

In [37]:
content_split_list = [record[0].split() for record in data]
len(content_split_list)

220

In [38]:
content_tokenized_list = tokenizer(
    content_split_list,
    is_split_into_words=True,  # True if a sequence is already split into words, input_shape = list[list[str]]
    max_length=MAX_SEQ_LENGTH,
    padding="max_length",
    truncation=True,
    add_special_tokens=True,   # add [CLS] at first of the sentence, [SEP] at the end of sentence
    return_tensors="tf",       # tf (tensorflow), pt (pytorch), np (numpy)
    return_attention_mask=True,
)


for k, v in content_tokenized_list.items():
    print(k)
    print(v)
    print('~'*100)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


input_ids
tf.Tensor(
[[  101 11113 24158 ...     0     0     0]
 [  101 21358 28029 ...     0     0     0]
 [  101 17712 19466 ...  2497  2487   102]
 ...
 [  101 15765 25632 ...     0     0     0]
 [  101 11748 27292 ... 27292 25457   102]
 [  101 10975  9648 ...     0     0     0]], shape=(220, 512), dtype=int32)
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
attention_mask
tf.Tensor(
[[1 1 1 ... 0 0 0]
 [1 1 1 ... 0 0 0]
 [1 1 1 ... 1 1 1]
 ...
 [1 1 1 ... 0 0 0]
 [1 1 1 ... 1 1 1]
 [1 1 1 ... 0 0 0]], shape=(220, 512), dtype=int32)
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [39]:
content_tokenized_list[0]

Encoding(num_tokens=512, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [40]:
print(content_split_list[0])

['Abhishek', 'Jha', 'Application', 'Development', 'Associate', '-', 'Accenture', 'Bengaluru,', 'Karnataka', '-', 'Email', 'me', 'on', 'Indeed:', 'indeed.com/r/Abhishek-Jha/10e7a8cb732bc43a', '•', 'To', 'work', 'for', 'an', 'organization', 'which', 'provides', 'me', 'the', 'opportunity', 'to', 'improve', 'my', 'skills', 'and', 'knowledge', 'for', 'my', 'individual', 'and', "company's", 'growth', 'in', 'best', 'possible', 'ways.', 'Willing', 'to', 'relocate', 'to:', 'Bangalore,', 'Karnataka', 'WORK', 'EXPERIENCE', 'Application', 'Development', 'Associate', 'Accenture', '-', 'November', '2017', 'to', 'Present', 'Role:', 'Currently', 'working', 'on', 'Chat-bot.', 'Developing', 'Backend', 'Oracle', 'PeopleSoft', 'Queries', 'for', 'the', 'Bot', 'which', 'will', 'be', 'triggered', 'based', 'on', 'given', 'input.', 'Also,', 'Training', 'the', 'bot', 'for', 'different', 'possible', 'utterances', '(Both', 'positive', 'and', 'negative),', 'which', 'will', 'be', 'given', 'as', 'input', 'by', 'the'

In [41]:
# Tokens generated from a sentence
print(len(content_tokenized_list[0].tokens))
print(content_tokenized_list[0].tokens)

512
['[CLS]', 'ab', '##his', '##he', '##k', 'j', '##ha', 'application', 'development', 'associate', '-', 'accent', '##ure', 'bengal', '##uru', ',', 'karnataka', '-', 'email', 'me', 'on', 'indeed', ':', 'indeed', '.', 'com', '/', 'r', '/', 'ab', '##his', '##he', '##k', '-', 'j', '##ha', '/', '10', '##e', '##7', '##a', '##8', '##cb', '##7', '##32', '##bc', '##43', '##a', '•', 'to', 'work', 'for', 'an', 'organization', 'which', 'provides', 'me', 'the', 'opportunity', 'to', 'improve', 'my', 'skills', 'and', 'knowledge', 'for', 'my', 'individual', 'and', 'company', "'", 's', 'growth', 'in', 'best', 'possible', 'ways', '.', 'willing', 'to', 'relocate', 'to', ':', 'bangalore', ',', 'karnataka', 'work', 'experience', 'application', 'development', 'associate', 'accent', '##ure', '-', 'november', '2017', 'to', 'present', 'role', ':', 'currently', 'working', 'on', 'chat', '-', 'bot', '.', 'developing', 'back', '##end', 'oracle', 'peoples', '##oft', 'que', '##ries', 'for', 'the', 'bot', 'which', '

In [42]:
# Map each token to an ID
print(len(content_tokenized_list[0].ids))
print(content_tokenized_list[0].ids)

512
[101, 11113, 24158, 5369, 2243, 1046, 3270, 4646, 2458, 5482, 1011, 9669, 5397, 8191, 14129, 1010, 12092, 1011, 10373, 2033, 2006, 5262, 1024, 5262, 1012, 4012, 1013, 1054, 1013, 11113, 24158, 5369, 2243, 1011, 1046, 3270, 1013, 2184, 2063, 2581, 2050, 2620, 27421, 2581, 16703, 9818, 23777, 2050, 1528, 2000, 2147, 2005, 2019, 3029, 2029, 3640, 2033, 1996, 4495, 2000, 5335, 2026, 4813, 1998, 3716, 2005, 2026, 3265, 1998, 2194, 1005, 1055, 3930, 1999, 2190, 2825, 3971, 1012, 5627, 2000, 20102, 2000, 1024, 14022, 1010, 12092, 2147, 3325, 4646, 2458, 5482, 9669, 5397, 1011, 2281, 2418, 2000, 2556, 2535, 1024, 2747, 2551, 2006, 11834, 1011, 28516, 1012, 4975, 2067, 10497, 14721, 7243, 15794, 10861, 5134, 2005, 1996, 28516, 2029, 2097, 2022, 13330, 2241, 2006, 2445, 7953, 1012, 2036, 1010, 2731, 1996, 28516, 2005, 2367, 2825, 14395, 26755, 1006, 2119, 3893, 1998, 4997, 1007, 1010, 2029, 2097, 2022, 2445, 2004, 7953, 2011, 1996, 5310, 1012, 2495, 1038, 1012, 1041, 1999, 2592, 2671, 1998, 

In [43]:
# The position of each token (a word can be split into multiple tokens) in the real sentence
print(len(content_tokenized_list[0].word_ids))
print(content_tokenized_list[0].word_ids)

512
[None, 0, 0, 0, 0, 1, 1, 2, 3, 4, 5, 6, 6, 7, 7, 7, 8, 9, 10, 11, 12, 13, 13, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 36, 36, 37, 38, 39, 40, 41, 41, 42, 43, 44, 45, 45, 46, 46, 47, 48, 49, 50, 51, 52, 53, 53, 54, 55, 56, 57, 58, 59, 59, 60, 61, 62, 63, 63, 63, 63, 64, 65, 65, 66, 67, 67, 68, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 79, 80, 80, 81, 82, 83, 84, 85, 86, 87, 87, 88, 88, 89, 90, 91, 91, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 100, 101, 102, 102, 102, 103, 104, 105, 106, 107, 108, 108, 108, 108, 108, 109, 110, 111, 112, 113, 114, 115, 115, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 134, 134, 135, 135, 135, 136, 137, 138, 139, 140, 141, 142, 143, 143, 144, 145, 146, 146, 146, 147, 148, 148, 149, 150, 151, 151, 151, 152, 153, 154, 154, 155, 156, 157, 157, 15

### Tokenize X + Create Dataset

In [44]:
def tokenize_content(tokenizer, content: list[list[str]], max_seq_length):
    content_tokenized = tokenizer(
        content, is_split_into_words=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        truncation=True,
        add_special_tokens=True,
        return_attention_mask=True,
        return_tensors="tf"
    )
    return content_tokenized

In [45]:
def align_label_to_token(content_tokenized: dict, entity_indices_padded: list[list[int]]):
    """
    Map labels from word level → token level, -100 for special tokens
    """

    entity_labels = []

    for i, enities_each_record in enumerate(entity_indices_padded):
        token_positions = content_tokenized[i].word_ids
        entity_labels_each_record = []

        for token_pos in token_positions:
            # if special tokens like [CLS], [SEP], [PAD]
            if token_pos == None:
              # then set -100 for Y to ignore compute loss for special tokens, tf.keras.losses.CrossEntropy(ignore_index=-100)
                entity_labels_each_record.append(-100)
            else:
                # token_pos = word_pos = entity_pos, each word mapped to an entity
                entity_labels_each_record.append(enities_each_record[token_pos])

        entity_labels.append(entity_labels_each_record)

    entity_labels = tf.constant(entity_labels, dtype=tf.int32)
    return entity_labels

In [46]:
content_tokenized = tokenize_content(tokenizer, content_split_list, max_seq_length=MAX_SEQ_LENGTH)
entity_labels = align_label_to_token(content_tokenized, entity_indices_padded)

entity_labels

<tf.Tensor: shape=(220, 512), dtype=int32, numpy=
array([[-100,    9,    9, ..., -100, -100, -100],
       [-100,    9,    9, ..., -100, -100, -100],
       [-100,    9,    9, ...,   11,   11, -100],
       ...,
       [-100,    9,    9, ..., -100, -100, -100],
       [-100,    9,    9, ...,   11,   11, -100],
       [-100,    9,    9, ..., -100, -100, -100]], dtype=int32)>

In [47]:
def create_tf_dataset(content_tokenized, entity_labels, batch_size=16, shuffle=True):
    input_ids = content_tokenized['input_ids']
    attention_mask = content_tokenized['attention_mask']

    dataset = tf.data.Dataset.from_tensor_slices((
        {"input_ids": input_ids, "attention_mask": attention_mask},
        entity_labels
    ))

    if shuffle:
        dataset = dataset.shuffle(buffer_size=input_ids.numpy().shape[0], reshuffle_each_iteration=True)

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

In [48]:
train_dataset = create_tf_dataset(content_tokenized, entity_labels)
train_dataset

<_PrefetchDataset element_spec=({'input_ids': TensorSpec(shape=(None, 512), dtype=tf.int32, name=None), 'attention_mask': TensorSpec(shape=(None, 512), dtype=tf.int32, name=None)}, TensorSpec(shape=(None, 512), dtype=tf.int32, name=None))>

In [49]:
len(train_dataset)

14

# Fine-tuning

In [50]:
from transformers import create_optimizer

In [51]:
N_BATCHES_PER_EPOCH = len(train_dataset)
N_EPOCHS = 20
LEARNING_RATE = 1e-3

optimizer, lr_schedule = create_optimizer(
    init_lr=LEARNING_RATE,
    num_warmup_steps=0,
    num_train_steps=N_BATCHES_PER_EPOCH * N_EPOCHS,
)


model.compile(
    optimizer=optimizer,
    loss=model.hf_compute_loss,
    metrics=['accuracy']
)


model.fit(train_dataset, epochs=N_EPOCHS)

Epoch 1/20
14/14 [==============================] - 24s 357ms/step - loss: 1.5479 - accuracy: 0.5799
Epoch 2/20
14/14 [==============================] - 5s 346ms/step - loss: 0.8493 - accuracy: 0.6648
Epoch 3/20
14/14 [==============================] - 5s 330ms/step - loss: 0.7571 - accuracy: 0.6715
Epoch 4/20
14/14 [==============================] - 5s 332ms/step - loss: 0.6904 - accuracy: 0.6747
Epoch 5/20
14/14 [==============================] - 5s 335ms/step - loss: 0.6427 - accuracy: 0.6777
Epoch 6/20
14/14 [==============================] - 5s 335ms/step - loss: 0.6120 - accuracy: 0.6817
Epoch 7/20
14/14 [==============================] - 5s 338ms/step - loss: 0.5875 - accuracy: 0.6866
Epoch 8/20
14/14 [==============================] - 5s 338ms/step - loss: 0.5673 - accuracy: 0.6903
Epoch 9/20
14/14 [==============================] - 5s 341ms/step - loss: 0.5508 - accuracy: 0.6921
Epoch 10/20
14/14 [==============================] - 5s 345ms/step - loss: 0.5376 - accuracy: 0.693

In [52]:
text = "Manisha Bharti. 3.5 years of professional IT experience in Banking and Finance domain"
inputs = tokenizer(
    text,
    is_split_into_words=False,
    max_length=512,
    padding="max_length",
    truncation=True,
    add_special_tokens=True,
    return_tensors="tf",
)

In [53]:
outputs = model(inputs)
outputs

TFTokenClassifierOutput(loss=None, logits=<tf.Tensor: shape=(1, 512, 12), dtype=float32, numpy=
array([[[-0.9772741 ,  0.26171586, -0.64110976, ...,  0.79659075,
         -0.8453577 ,  0.14403002],
        [-0.997256  ,  1.5296131 , -2.1200345 , ...,  3.241558  ,
         -2.195953  ,  0.669399  ],
        [-1.031927  ,  0.768508  , -1.608252  , ...,  1.311802  ,
         -3.396857  ,  1.8082818 ],
        ...,
        [-0.58536226, -0.04853547,  0.03321726, ..., -1.5606953 ,
         -1.4928701 ,  1.0702683 ],
        [-0.63254505, -0.06652445, -0.07729147, ..., -1.4849029 ,
         -1.4900881 ,  1.1482655 ],
        [-0.5834423 , -0.01427586, -0.06113869, ..., -1.4605342 ,
         -1.624383  ,  1.1285161 ]]], dtype=float32)>, hidden_states=None, attentions=None)

In [54]:
pred_entity_ids = tf.argmax(outputs.logits, axis=-1).numpy()[0]
pred_entity_ids

array([ 6,  9, 11,  9,  9, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
       11, 11, 11, 11, 11, 11, 11, 11, 11,  9, 11, 11, 11, 11, 11, 11, 11,
       11, 11, 11,  7, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
       11, 11, 11, 11, 11,  9,  9,  9,  9, 11, 11, 11, 11, 11, 11, 11, 11,
       11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
       11,  9,  9,  9, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
       11, 11, 11, 11, 11, 11, 11, 11, 11,  9,  9,  9,  9,  1, 11, 11, 11,
       11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,  9,  1,  1,  9, 11, 11,
       11, 11, 11, 11, 11, 11, 11, 11, 11,  9, 11, 11, 11, 11, 11, 11, 11,
        7, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
       11, 11,  9, 11, 11, 11,  9,  9,  9,  9, 11, 11, 11, 11, 11, 11, 11,
       11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
       11, 11,  1, 11, 11, 11, 11, 11,  9,  9,  9,  9, 11, 11, 11, 11, 11,
       11, 11, 11, 11, 11

In [55]:
pred_entites = [idx_to_entity[idx] for idx in pred_entity_ids]

In [56]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for token, entity in zip(tokens, pred_entites):
    if not token == "[PAD]":
      print(f"{token:15} -> {entity}")

[CLS]           -> Degree
mani            -> Name
##sha           -> Empty
b               -> Name
##hart          -> Name
##i             -> Empty
.               -> Empty
3               -> Empty
.               -> Empty
5               -> Empty
years           -> Empty
of              -> Empty
professional    -> Empty
it              -> Empty
experience      -> Empty
in              -> Empty
banking         -> Empty
and             -> Empty
finance         -> Empty
domain          -> Empty
[SEP]           -> Empty


**Due to imbalance classes** 😞



